In [4]:
from datetime import date
from pathlib import Path

import blpapi
import pandas as pd

SECURITY = ["AAPL US Equity"]
FIELD = ["PX_LAST"]
START_DATE = date(2026, 1, 1)
END_DATE = date(2026, 8, 19)
DATA_DIR = Path("data")


def _element_value(row, field):
    if not row.hasElement(field) or row.getElement(field).isNull():
        return None
    return row.getElementAsFloat(field)


def fetch_historical_prices(
    securities=SECURITY,
    fields=FIELD,
    start_date=START_DATE,
    end_date=END_DATE,
):
    """Pull daily Bloomberg history for every security and field provided."""
    if not securities:
        raise ValueError("securities must contain at least one Bloomberg security")
    if not fields:
        raise ValueError("fields must contain at least one Bloomberg field")
    if not isinstance(start_date, date) or not isinstance(end_date, date):
        raise TypeError("start_date and end_date must be datetime.date objects")
    if start_date > end_date:
        raise ValueError("start_date must be on or before end_date")

    session = blpapi.Session()
    if not session.start():
        raise RuntimeError("Could not start Bloomberg session. Is Bloomberg Terminal running?")

    try:
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Could not open Bloomberg reference-data service.")

        request = session.getService("//blp/refdata").createRequest("HistoricalDataRequest")
        for security in securities:
            request.getElement("securities").appendValue(security)
        for field in fields:
            request.getElement("fields").appendValue(field)
        request.set("startDate", start_date.strftime("%Y%m%d"))
        request.set("endDate", end_date.strftime("%Y%m%d"))
        request.set("periodicitySelection", "DAILY")
        session.sendRequest(request)

        records = []
        while True:
            event = session.nextEvent()
            for message in event:
                if not message.hasElement("securityData"):
                    continue
                security_data = message.getElement("securityData")
                security = security_data.getElementAsString("security")
                field_data = security_data.getElement("fieldData")
                for index in range(field_data.numValues()):
                    row = field_data.getValueAsElement(index)
                    record = {
                        "security": security,
                        "date": row.getElementAsDatetime("date"),
                    }
                    record.update({field.lower(): _element_value(row, field) for field in fields})
                    records.append(record)
            if event.eventType() == blpapi.Event.RESPONSE:
                break

        if not records:
            return pd.DataFrame(columns=["security", "date", *[field.lower() for field in fields]])
        return pd.DataFrame(records).set_index(["security", "date"]).sort_index()
    finally:
        session.stop()


prices = fetch_historical_prices(
    securities=SECURITY,
    fields=FIELD,
    start_date=START_DATE,
    end_date=END_DATE,
)
DATA_DIR.mkdir(parents=True, exist_ok=True)
output_path = DATA_DIR / "historical_prices.csv"
prices.to_csv(output_path)
print(f"Saved {len(prices):,} rows for {len(SECURITY)} security to {output_path}")
prices.tail()


Saved 158 rows for 1 security to data\historical_prices.csv


px_last
security       date               
AAPL US Equity 2026-08-13   305.26
               2026-08-14   305.93
               2026-08-17   305.59
               2026-08-18   310.03
               2026-08-19   315.69